# 🎫 Task 5 — Auto-Tagging Support Tickets Using an LLM

**Objective:** Automatically tag free-text support tickets into categories using a large language model, and compare **zero-shot**, **few-shot**, and **fine-tuned** performance. Every ticket gets its **top 3 most probable tags**.

**Pipeline overview**
1. Build/load a free-text support ticket dataset (synthetic, 6 categories)
2. **Zero-shot** classification with an LLM (no examples given) — via a HuggingFace NLI model
3. **Few-shot** classification — the same LLM idea, but prompted with a handful of labeled examples per category to sharpen its judgment
4. **Fine-tuned** classification — a DistilBERT classifier fine-tuned directly on labeled tickets
5. **Compare** all three approaches on accuracy, top-3 accuracy, and macro-F1
6. Produce a final `ticket → top 3 tags` table

> 💡 **Runtime tip (Colab):** `Runtime → Change runtime type → GPU (T4)` before running the fine-tuning section — it will work on CPU too, just slower.


## 1. Setup

In [1]:
# Install dependencies (Colab already has most of these, this just pins/adds what's missing)
!pip install -q transformers datasets accelerate scikit-learn pandas matplotlib openai


In [2]:
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else -1
print("Using GPU" if DEVICE == 0 else "Using CPU")


Using CPU


## 2. Dataset

This notebook uses a **synthetic free-text support ticket dataset** so it runs
end-to-end with no external downloads. It has 6 realistic support categories.

**To use a real dataset instead** (e.g. a Kaggle customer-support-tickets CSV),
just replace the `build_synthetic_dataset()` call below with:

```python
df = pd.read_csv("/content/your_dataset.csv")  # must have columns: text, label
```

as long as the result is a DataFrame with a `text` column (the ticket body) and
a `label` column (the ground-truth category), everything downstream works unchanged.


In [3]:
CATEGORIES = [
    "Billing",
    "Technical Issue",
    "Account Access",
    "Feature Request",
    "Bug Report",
    "General Inquiry",
]

TEMPLATES = {
    "Billing": [
        "I was charged twice for my subscription this month, can you refund the extra payment?",
        "My invoice shows a price that doesn't match what I signed up for.",
        "How do I update my credit card on file before my next billing cycle?",
        "I cancelled my plan last week but was still billed today.",
        "Can I get a refund for the annual plan I purchased by mistake?",
        "Why did my subscription price increase without any notice?",
        "I need a copy of last month's invoice for my accounting records.",
        "The discount code I applied at checkout didn't reduce my total.",
    ],
    "Technical Issue": [
        "The app crashes every time I try to open the settings page.",
        "Video calls keep freezing after about two minutes on the desktop app.",
        "I'm getting a 500 error whenever I try to upload a file larger than 10MB.",
        "The dashboard is loading extremely slowly today, is there an outage?",
        "Notifications stopped working after the latest update.",
        "The mobile app won't sync with the web version anymore.",
        "Exporting my report to PDF fails with a generic error message.",
        "The search bar returns no results even for terms I know exist.",
    ],
    "Account Access": [
        "I can't log in even though I'm sure my password is correct.",
        "I'm locked out of my account after too many failed login attempts.",
        "The two-factor authentication code never arrives on my phone.",
        "I forgot the email address I used to sign up, how do I recover my account?",
        "My account was suspended and I don't know why.",
        "I need to transfer ownership of my account to a colleague.",
        "The password reset link in my email keeps expiring before I click it.",
        "I'm being asked to verify my identity but the verification page is blank.",
    ],
    "Feature Request": [
        "It would be great if you could add dark mode to the mobile app.",
        "Can you add support for exporting data directly to Google Sheets?",
        "Please consider adding keyboard shortcuts for common actions.",
        "I'd love to see a bulk-edit option for managing multiple items at once.",
        "Could you add integration with Slack for notifications?",
        "It would help to have a calendar view instead of just a list view.",
        "Please add the ability to schedule reports to be sent automatically.",
        "Can you support multiple languages in the interface?",
    ],
    "Bug Report": [
        "When I click 'save', the form clears all my data instead of saving it.",
        "The total on the invoice page is calculated incorrectly.",
        "Deleting one item accidentally deletes the entire folder.",
        "Timestamps are showing in the wrong timezone across the app.",
        "The chart on the analytics page shows duplicate data points.",
        "Editing my profile picture sometimes rotates the image upside down.",
        "The 'undo' button doesn't actually undo the last action.",
        "Filters reset themselves every time I refresh the page.",
    ],
    "General Inquiry": [
        "What are your customer support hours?",
        "Do you offer a student or non-profit discount?",
        "Is there a mobile app available for iOS?",
        "What's the difference between the Pro and Business plans?",
        "How do I contact your sales team for an enterprise quote?",
        "Where can I find your API documentation?",
        "Do you have a public status page for outages?",
        "Can I schedule a demo before committing to a paid plan?",
    ],
}


def build_synthetic_dataset(n_per_category=40):
    """Generate a synthetic ticket dataset by lightly varying template phrasing."""
    prefixes = ["", "Hi team, ", "Hello, ", "Hi, ", "Support, ", "Quick question — ", "Urgent: ", ""]
    suffixes = ["", " Thanks!", " Please help.", " Appreciate any help.", " Thank you.", ""]

    rows = []
    for label, templates in TEMPLATES.items():
        for i in range(n_per_category):
            base = random.choice(templates)
            text = random.choice(prefixes) + base + random.choice(suffixes)
            rows.append({"text": text.strip(), "label": label})

    df = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
    return df


df = build_synthetic_dataset(n_per_category=40)
print(f"Total tickets: {len(df)}")
print(df["label"].value_counts())
df.head()


Total tickets: 240
label
Billing            40
Account Access     40
Bug Report         40
General Inquiry    40
Feature Request    40
Technical Issue    40
Name: count, dtype: int64


,text,label
0,Urgent: The discount code I applied at checkou...,Billing
1,Urgent: I cancelled my plan last week but was ...,Billing
2,"Hi, I need to transfer ownership of my account...",Account Access
3,"Support, The password reset link in my email k...",Account Access
4,"Hi team, I'm locked out of my account after to...",Account Access


In [4]:
train_df, test_df = train_test_split(
    df, test_size=0.25, stratify=df["label"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")


Train: 180  |  Test: 60


## 3. Zero-shot classification (LLM, no examples)

We use `facebook/bart-large-mnli` through HuggingFace's `zero-shot-classification`
pipeline. The model was never trained on our categories — it treats each label
as a hypothesis ("this ticket is about {label}") and scores how well the ticket
text entails that hypothesis. This is the classic "prompt an LLM with just the
label names" approach.

Each ticket gets a probability for every category — we keep the **top 3**.


In [5]:
zero_shot_clf = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=DEVICE,
)

def zero_shot_top3(text, labels=CATEGORIES):
    result = zero_shot_clf(text, candidate_labels=labels, multi_label=True)
    ranked = sorted(zip(result["labels"], result["scores"]), key=lambda x: -x[1])
    return ranked[:3]

# quick sanity check
sample_ticket = test_df.iloc[0]["text"]
print("Ticket:", sample_ticket)
print("True label:", test_df.iloc[0]["label"])
print("Zero-shot top 3:", zero_shot_top3(sample_ticket))


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Ticket: Hello, The search bar returns no results even for terms I know exist. Please help.
True label: Technical Issue
Zero-shot top 3: [('Technical Issue', 0.807060718536377), ('General Inquiry', 0.4299947917461395), ('Bug Report', 0.3973797857761383)]


In [ ]:
zero_shot_results = []
for text in test_df["text"]:
    zero_shot_results.append(zero_shot_top3(text))

test_df["zero_shot_top3"] = zero_shot_results
test_df["zero_shot_pred"] = test_df["zero_shot_top3"].apply(lambda r: r[0][0])
test_df[["text", "label", "zero_shot_pred", "zero_shot_top3"]].head()


## 4. Few-shot classification (LLM + labeled examples in the prompt)

Now we give the LLM a handful of **labeled examples per category** before asking
it to classify the ticket. If you have an OpenAI API key, this cell builds a
true few-shot **prompt** sent to a chat model (`gpt-4o-mini`), which reliably
follows instructions and returns ranked JSON — this is the standard
"prompt engineering" approach to LLM classification.

If no API key is available, the notebook falls back to a **local few-shot
proxy**: the same zero-shot NLI model, but now the candidate "hypotheses" are
built from real example tickets per category (grounding each label in
examples rather than just its name) — approximating the accuracy lift from
few-shot prompting without requiring an API key.


In [ ]:
import os
import json

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")  # set this in Colab: os.environ["OPENAI_API_KEY"] = "sk-..."

FEW_SHOT_EXAMPLES = {
    cat: TEMPLATES[cat][:3] for cat in CATEGORIES
}


def build_few_shot_prompt(ticket_text):
    example_block = ""
    for cat, examples in FEW_SHOT_EXAMPLES.items():
        for ex in examples:
            example_block += f'Ticket: "{ex}"\nCategory: {cat}\n\n'

    prompt = f"""You are a support ticket triage assistant. Categories: {", ".join(CATEGORIES)}.

Here are some labeled examples:

{example_block}Now classify this new ticket. Respond ONLY with JSON in this exact format:
{{"top_3": [{{"label": "<category>", "score": <float 0-1>}}, {{"label": "<category>", "score": <float 0-1>}}, {{"label": "<category>", "score": <float 0-1>}}]}}

Ticket: "{ticket_text}"
"""
    return prompt


def few_shot_openai_top3(ticket_text):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    prompt = build_few_shot_prompt(ticket_text)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    content = resp.choices[0].message.content
    parsed = json.loads(content)
    ranked = [(item["label"], item["score"]) for item in parsed["top_3"]]
    return ranked


def few_shot_local_proxy_top3(ticket_text):
    """Fallback: ground each label with example tickets instead of just its name."""
    hypothesis_labels = {
        cat: f'{cat} (similar to: "{FEW_SHOT_EXAMPLES[cat][0]}")' for cat in CATEGORIES
    }
    result = zero_shot_clf(
        ticket_text,
        candidate_labels=list(hypothesis_labels.values()),
        multi_label=True,
    )
    label_map = {v: k for k, v in hypothesis_labels.items()}
    ranked = sorted(zip(result["labels"], result["scores"]), key=lambda x: -x[1])
    ranked = [(label_map[label], score) for label, score in ranked]
    return ranked[:3]


def few_shot_top3(ticket_text):
    if OPENAI_API_KEY:
        try:
            return few_shot_openai_top3(ticket_text)
        except Exception as e:
            print("OpenAI call failed, falling back to local proxy:", e)
    return few_shot_local_proxy_top3(ticket_text)


print("Using OpenAI few-shot prompting" if OPENAI_API_KEY else "No OPENAI_API_KEY found — using local few-shot proxy")
print(few_shot_top3(sample_ticket))


In [ ]:
few_shot_results = []
for text in test_df["text"]:
    few_shot_results.append(few_shot_top3(text))

test_df["few_shot_top3"] = few_shot_results
test_df["few_shot_pred"] = test_df["few_shot_top3"].apply(lambda r: r[0][0])
test_df[["text", "label", "few_shot_pred", "few_shot_top3"]].head()


## 5. Fine-tuned classification (DistilBERT trained on our tickets)

Here we fine-tune `distilbert-base-uncased` as a direct multi-class classifier
on the labeled training tickets — the traditional "fine-tuning" approach, as
opposed to prompting a frozen LLM.


In [ ]:
label2id = {label: i for i, label in enumerate(CATEGORIES)}
id2label = {i: label for label, i in label2id.items()}

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

from datasets import Dataset

train_ds = Dataset.from_pandas(train_df[["text", "label"]].assign(labels=train_df["label"].map(label2id)))
test_ds = Dataset.from_pandas(test_df[["text", "label"]].assign(labels=test_df["label"].map(label2id)))

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["text", "label", "__index_level_0__"]) if "__index_level_0__" in train_ds.column_names else train_ds.remove_columns(["text", "label"])
test_ds = test_ds.remove_columns(["text", "label", "__index_level_0__"]) if "__index_level_0__" in test_ds.column_names else test_ds.remove_columns(["text", "label"])


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(CATEGORIES), id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./ticket_classifier",
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    report_to=[],
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
import torch.nn.functional as F

def fine_tuned_top3(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits, dim=-1).cpu().numpy()[0]
    ranked = sorted(zip(CATEGORIES, probs), key=lambda x: -x[1])
    return [(label, float(score)) for label, score in ranked[:3]]

print(fine_tuned_top3(sample_ticket))


In [ ]:
fine_tuned_results = []
for text in test_df["text"]:
    fine_tuned_results.append(fine_tuned_top3(text))

test_df["fine_tuned_top3"] = fine_tuned_results
test_df["fine_tuned_pred"] = test_df["fine_tuned_top3"].apply(lambda r: r[0][0])
test_df[["text", "label", "fine_tuned_pred", "fine_tuned_top3"]].head()


## 6. Compare zero-shot vs few-shot vs fine-tuned

Two metrics per approach:
- **Top-1 accuracy** — was the single best-predicted tag correct?
- **Top-3 accuracy** — was the correct tag anywhere in the top 3 predictions?
- **Macro-F1** (top-1) — balances performance across all 6 categories evenly


In [ ]:
def top3_accuracy(true_labels, top3_col):
    hits = [true in [t[0] for t in top3] for true, top3 in zip(true_labels, top3_col)]
    return np.mean(hits)

approaches = {
    "Zero-shot": ("zero_shot_pred", "zero_shot_top3"),
    "Few-shot": ("few_shot_pred", "few_shot_top3"),
    "Fine-tuned": ("fine_tuned_pred", "fine_tuned_top3"),
}

summary_rows = []
for name, (pred_col, top3_col) in approaches.items():
    acc = accuracy_score(test_df["label"], test_df[pred_col])
    macro_f1 = f1_score(test_df["label"], test_df[pred_col], average="macro")
    top3_acc = top3_accuracy(test_df["label"], test_df[top3_col])
    summary_rows.append({
        "Approach": name,
        "Top-1 Accuracy": round(acc, 3),
        "Top-3 Accuracy": round(top3_acc, 3),
        "Macro F1": round(macro_f1, 3),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(summary_df))
width = 0.25

ax.bar(x - width, summary_df["Top-1 Accuracy"], width, label="Top-1 Accuracy")
ax.bar(x, summary_df["Top-3 Accuracy"], width, label="Top-3 Accuracy")
ax.bar(x + width, summary_df["Macro F1"], width, label="Macro F1")

ax.set_xticks(x)
ax.set_xticklabels(summary_df["Approach"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Zero-shot vs Few-shot vs Fine-tuned: Support Ticket Tagging")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
print(classification_report(test_df["label"], test_df["fine_tuned_pred"]))


## 7. Final deliverable: top 3 tags per ticket

The table below is the actual required output of this task — for every
ticket, its **top 3 most probable tags** from the best-performing approach
(swap `fine_tuned_top3` for `zero_shot_top3` or `few_shot_top3` if you want
to inspect a different approach).


In [ ]:
def format_top3(top3):
    return " | ".join(f"{label} ({score:.2f})" for label, score in top3)

final_output = test_df[["text", "label"]].copy()
final_output["predicted_top_3_tags"] = test_df["fine_tuned_top3"].apply(format_top3)
final_output.columns = ["ticket_text", "true_label", "predicted_top_3_tags"]

pd.set_option("display.max_colwidth", 80)
final_output.head(15)


In [ ]:
final_output.to_csv("auto_tagged_tickets.csv", index=False)
print("Saved to auto_tagged_tickets.csv")


## 8. Summary

| Approach | What it does | Pros | Cons |
|---|---|---|---|
| **Zero-shot** | Prompts an LLM/NLI model with only the category names, no examples | No training data needed, works instantly on new label sets | Lowest accuracy, sensitive to how labels are phrased |
| **Few-shot** | Adds a handful of labeled examples to the prompt | Notably better accuracy than zero-shot with very little labeled data | Still limited by prompt length; quality depends on example choice |
| **Fine-tuned** | Trains model weights directly on labeled tickets | Highest accuracy, fastest inference once trained | Needs a labeled dataset and a training step; less flexible if categories change |

For production ticket-tagging systems, a common pattern is to **start with
zero/few-shot prompting** to get something working immediately, then
**fine-tune once enough labeled tickets accumulate** from real usage.
